In [1]:
!pip install -q sentence-transformers faiss-cpu pypdf python-docx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.5 MB/s eta 0:00:00


In [5]:
import os
import faiss
import numpy as np
from google.colab import files
from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer

# Upload document
uploaded = files.upload()
file_name = list(uploaded.keys())[0]

# Extract text
def extract_text(file_name):
    extension = os.path.splitext(file_name)[1].lower()

    if extension == ".pdf":
        reader = PdfReader(file_name)
        return "\n".join(page.extract_text() or "" for page in reader.pages)

    elif extension == ".docx":
        document = Document(file_name)
        return "\n".join(paragraph.text for paragraph in document.paragraphs)

    elif extension == ".txt":
        with open(file_name, "r", encoding="utf-8") as file:
            return file.read()

    else:
        raise ValueError("Only PDF, DOCX, and TXT files are supported.")

text = extract_text(file_name)

print("Text extracted successfully!")

# Split text into chunks
chunk_size = 500

chunks = [
    text[i:i + chunk_size]
    for i in range(0, len(text), chunk_size)
]

print("Number of chunks:", len(chunks))

# Load multilingual Sentence Transformer
model = SentenceTransformer(
    "paraphrase-multilingual-MiniLM-L12-v2"
)

# Create multilingual embeddings
embeddings = model.encode(
    chunks,
    convert_to_numpy=True
).astype("float32")

print("Embedding dimension:", embeddings.shape[1])

# Normalize embeddings
faiss.normalize_L2(embeddings)

# Store embeddings in FAISS
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Multilingual embeddings stored in FAISS!")

# Ask questions
while True:

    question = input("\nAsk a question in any language (type 'exit' to stop): ")

    if question.lower() == "exit":
        print("Application Closed.")
        break

    # Convert question into multilingual embedding
    question_embedding = model.encode(
        [question],
        convert_to_numpy=True
    ).astype("float32")

    faiss.normalize_L2(question_embedding)

    # Find most relevant chunk
    scores, indices = index.search(
        question_embedding,
        k=1
    )

    relevant_chunk = chunks[indices[0][0]]
    similarity_score = scores[0][0]

    # Display answer
    print("\nAnswer:")
    print(relevant_chunk)

    print("\nSimilarity Score:",
          round(float(similarity_score), 3))

Saving Case study.docx to Case study.docx
Text extracted successfully!
Number of chunks: 41


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embedding dimension: 384
Multilingual embeddings stored in FAISS!

Ask a question in any language (type 'exit' to stop): what module does this taks about?

Answer:
irectly on the fly without storing any matrix.
Implementation Code:
import openmdao.api as om
import numpy as np
# Matrix-free component computing directional matrix-vector products
class MatrixFreeSquareComp(om.ExplicitComponent):
    def setup(self):
        # 100,000 element vector
        self.add_input('x', val=np.ones(100000))
        self.add_output('y', val=np.ones(100000))
        self.declare_partials(of='y', wrt='x')
    def compute(self, inputs, outputs):
        outputs['y'] = input

Similarity Score: 0.555

Ask a question in any language (type 'exit' to stop): wht is openmdo

Answer:
with thousands of design variables in seconds. By bridging the gap between domain-specific engineering solvers and numerical optimization, OpenMDAO serves as a powerful, NASA-grade tool for modern multidisciplinary system design.
2

Using class object

In [6]:
import os
import faiss
from google.colab import files
from pypdf import PdfReader
from docx import Document
from sentence_transformers import SentenceTransformer


class MultilingualQA:

    def __init__(self):
        self.model = SentenceTransformer(
            "paraphrase-multilingual-MiniLM-L12-v2"
        )
        self.chunks = []
        self.index = None

    def upload_document(self):
        uploaded = files.upload()
        self.file_name = list(uploaded.keys())[0]
        print("Uploaded:", self.file_name)

    def extract_text(self):
        extension = os.path.splitext(self.file_name)[1].lower()

        if extension == ".pdf":
            reader = PdfReader(self.file_name)
            self.text = "\n".join(
                page.extract_text() or "" for page in reader.pages
            )

        elif extension == ".docx":
            document = Document(self.file_name)
            self.text = "\n".join(
                paragraph.text for paragraph in document.paragraphs
            )

        elif extension == ".txt":
            with open(self.file_name, "r", encoding="utf-8") as file:
                self.text = file.read()

        else:
            raise ValueError("Only PDF, DOCX and TXT are supported.")

        print("Text extracted successfully!")

    def create_embeddings(self):
        chunk_size = 500

        self.chunks = [
            self.text[i:i + chunk_size]
            for i in range(0, len(self.text), chunk_size)
        ]

        embeddings = self.model.encode(
            self.chunks,
            convert_to_numpy=True
        ).astype("float32")

        faiss.normalize_L2(embeddings)

        self.index = faiss.IndexFlatIP(embeddings.shape[1])
        self.index.add(embeddings)

        print("Chunks:", len(self.chunks))
        print("Embedding dimension:", embeddings.shape[1])
        print("Embeddings stored in FAISS!")

    def ask_question(self):

        while True:

            question = input(
                "\nAsk a question in any language (type 'exit' to stop): "
            )

            if question.lower() == "exit":
                print("Application Closed.")
                break

            question_embedding = self.model.encode(
                [question],
                convert_to_numpy=True
            ).astype("float32")

            faiss.normalize_L2(question_embedding)

            scores, indices = self.index.search(
                question_embedding,
                k=1
            )

            answer = self.chunks[indices[0][0]]

            print("\nAnswer:")
            print(answer)

            print("\nSimilarity Score:",
                  round(float(scores[0][0]), 3))


qa = MultilingualQA()

qa.upload_document()
qa.extract_text()
qa.create_embeddings()
qa.ask_question()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saving flow.txt to flow.txt
Uploaded: flow.txt
Text extracted successfully!
Chunks: 9
Embedding dimension: 384
Embeddings stored in FAISS!

Ask a question in any language (type 'exit' to stop): what is in this file

Answer:

                           │                 │
                     Entropy OK       Entropy Low
                           │                 │
                           ▼                 ▼
                Deliver Random      Initialize Next
                Bitstream to User   Sliding Window
                                             │
                                             ▼
                                   Read Current Sample
```

---

## Flow Summary

1. Acquire Laser OFF samples.

Similarity Score: 0.248

Ask a question in any language (type 'exit' to stop): give me the flow summary fully

Answer:

                           │                 │
                     Entropy OK       Entropy Low
                           │                 │
          

1. Objective

Build a multilingual Question Answering system that retrieves relevant information from uploaded PDF, DOCX, or TXT documents.

2. Document Processing

The uploaded document is processed by extracting its text and splitting the text into smaller chunks.

3. Multilingual Embeddings

The paraphrase-multilingual-MiniLM-L12-v2 Sentence Transformer converts document chunks and user questions into multilingual vector embeddings.

4. FAISS Similarity Search

FAISS stores the embeddings and performs cosine similarity search to find the most relevant document chunk.

5. Class and Object

The MultilingualQA class organizes the complete system, while the object created from the class performs document upload, text processing, embedding generation, and question answering.